# Telecom Customer Retention & Risk ETL Pipeline
**Objective:** Extract cleaned SQL views (`vw_ChurnData` and `vw_JoinData`), perform Pandas data quality checks, and flag high-flight-risk accounts for call-center prioritization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# 1. EXTRACT: Load historical customer view
file_path = "../Prediction data.xlsx"
data = pd.read_excel(file_path, sheet_name='vw_ChurnData')

# 2. TRANSFORM & DATA QUALITY CHECKS (Pandas ETL)
print(f"Dataset Shape: {data.shape}")
print(f"Duplicate Customer_IDs: {data['Customer_ID'].duplicated().sum()}")

# Fix negative Monthly_Charge anomalies
neg_count = (data['Monthly_Charge'] < 0).sum()
data['Monthly_Charge'] = data['Monthly_Charge'].abs()
print(f"Negative Monthly_Charge records fixed: {neg_count}")

# Fill missing categorical values
data['Value_Deal'] = data['Value_Deal'].fillna('None')
data['Internet_Type'] = data['Internet_Type'].fillna('None')

Dataset Shape: (6007, 32)
Duplicate Customer_IDs: 0
Negative Monthly_Charge records fixed: 11


In [2]:
# Drop identifier and post-churn columns not used for risk scoring
data = data.drop(['Customer_ID', 'Churn_Category', 'Churn_Reason'], axis=1)

columns_to_encode = [
    'Gender', 'Married', 'State', 'Value_Deal', 'Phone_Service', 'Multiple_Lines',
    'Internet_Service', 'Internet_Type', 'Online_Security', 'Online_Backup',
    'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV', 'Streaming_Movies',
    'Streaming_Music', 'Unlimited_Data', 'Contract', 'Paperless_Billing', 'Payment_Method'
]

label_encoders = {}
for column in columns_to_encode:
    label_encoders[column] = LabelEncoder()
    data[column] = label_encoders[column].fit_transform(data[column].astype(str))

data['Customer_Status'] = data['Customer_Status'].map({'Stayed': 0, 'Churned': 1})

X = data.drop('Customer_Status', axis=1)
y = data['Customer_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
# Train Random Forest with balanced class weights
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[793  50]
 [135 224]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.94      0.90       843
           1       0.82      0.62      0.71       359

    accuracy                           0.85      1202
   macro avg       0.84      0.78      0.80      1202
weighted avg       0.84      0.85      0.84      1202


In [4]:
# 3. LOAD: Score New Joiners Cohort and Export for Power BI Dashboard
new_data = pd.read_excel(file_path, sheet_name='vw_JoinData')
new_data['Monthly_Charge'] = new_data['Monthly_Charge'].abs()
new_data['Value_Deal'] = new_data['Value_Deal'].fillna('None')
new_data['Internet_Type'] = new_data['Internet_Type'].fillna('None')

original_data = new_data.copy()
new_data = new_data.drop(['Customer_ID', 'Customer_Status', 'Churn_Category', 'Churn_Reason'], axis=1)

for column in new_data.select_dtypes(include=['object']).columns:
    new_data[column] = label_encoders[column].transform(new_data[column].astype(str))

original_data['Customer_Status_Predicted'] = rf_model.predict(new_data)

# Add Value_Tier in Python ETL for Power BI reporting
original_data['Value_Tier'] = np.select(
    [
        original_data['Monthly_Charge'] >= 80,
        (original_data['Monthly_Charge'] >= 40) & (original_data['Monthly_Charge'] < 80)
    ],
    ['High Value (₹80+)', 'Mid Value (₹40-₹79)'],
    default='Low Value (<₹40)'
)

at_risk_customers = original_data[original_data['Customer_Status_Predicted'] == 1]
print(f"Total High-Risk Accounts Flagged for Call Center: {len(at_risk_customers)}")

at_risk_customers.to_csv("../Predictions.csv", index=False)

Total High-Risk Accounts Flagged for Call Center: 382
